# Berka synthetic data with RelDiff

This notebook adapts the preprocessed multi-table Berka data used by the overdraft experiment to RelDiff's expected `data/original/<dataset>/` layout, creates Syntherela metadata for the relational schema, then trains and samples with RelDiff.

The source is the existing `berka2` dataset, not the feature-engineered transaction export. This keeps the initial comparison focused on replacing ClavaDDPM with RelDiff. It includes all eight tables and their primary/foreign-key relationships. `trans.balance` remains in the generated table, so the overdraft target can still be derived as `balance < 0` by the transfer evaluator.

## RelDiff behavior and requirements

- The repository's training and sampling scripts require a CUDA-capable PyTorch environment: both construct the model with `.cuda()` even though their CLI exposes a device option. The source checkout also needs its dependencies installed and the project installed in editable mode (`pip install -e .`).
- Train for **more than 4,000 epochs**. The trainer only writes the `best_model` checkpoint required by the sampling script after epoch 4,000. The default below is 5,000; this is a functional baseline, not a claim of converged or tuned quality.
- Sampling with the default `--structure original` retains the original graph structure and table cardinalities. In particular, this script does not offer a direct transaction-row-count parameter. Changing the number of generated transactions requires RelDiff's generated-structure workflow or a deliberate code/configuration extension; do not interpret a larger sampling batch size as a larger dataset.
- RelDiff jointly generates table attributes conditioned on the relational graph. It does not enforce that derived values such as an overdraft-history feature are logically recomputed from generated transactions. This notebook does not include those engineered columns.

The RelDiff README describes the implementation as still being cleaned up, so treat this as an experimental integration and inspect all generated tables and relationships before downstream evaluation.

In [2]:
from datetime import datetime
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

import pandas as pd


def find_reldiff_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src/scripts/train_joint_diffusion.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find the RelDiff checkout from the notebook working directory.")


RELDIFF_ROOT = find_reldiff_root(Path("/home/coder/RelDiff"))
BERKA_SOURCE = Path(
    "/home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka2"
)
DATASET_NAME = "berka_overdraft"
ORIGINAL_DIR = RELDIFF_ROOT / "data/original" / DATASET_NAME
DATASET_CONFIG = RELDIFF_ROOT / "src/reldiff/configs/data/berka.toml"
MODEL_CONFIG = RELDIFF_ROOT / "src/reldiff/configs/reldiff_config.toml"
RUN_ID = datetime.now().strftime("reldiff_%Y%m%d_%H%M%S")

print("RelDiff root:", RELDIFF_ROOT)
print("Berka source:", BERKA_SOURCE)
print("Dataset name:", DATASET_NAME)
print("Run ID:", RUN_ID)

RelDiff root: /home/coder/RelDiff
Berka source: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka2
Dataset name: berka_overdraft
Run ID: reldiff_20261029_140854


## Check the runtime

Create/activate the environment described by `reldiff.yaml`, install this checkout with `pip install -e .`, and select that environment as the notebook kernel before continuing. This cell intentionally stops early if the runtime is not ready.

In [3]:
required_modules = ["torch", "torch_geometric", "syntherela", "wandb"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError(
        f"Missing RelDiff dependencies: {missing_modules}. Install the RelDiff environment and run `pip install -e .`."
    )

import torch
from syntherela.data import save_tables
from syntherela.metadata import Metadata

if not torch.cuda.is_available():
    raise RuntimeError(
        "The checked-in RelDiff train/sample scripts call `.cuda()` directly; use a CUDA-enabled kernel."
    )

if not BERKA_SOURCE.is_dir():
    raise FileNotFoundError(f"Berka source directory not found: {BERKA_SOURCE}")
if not MODEL_CONFIG.is_file() or not DATASET_CONFIG.is_file():
    raise FileNotFoundError("A required RelDiff TOML configuration file is missing.")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda, "| device:", torch.cuda.get_device_name(0))

PyTorch: 2.0.1+cu117
CUDA: 11.7 | device: NVIDIA L4


## Load Berka tables and declare the relational schema

The `berka2` CSVs store category codes as integers and dates as integer offsets, with column types recorded in each `*_domain.json`. The code below uses those domains for categorical-vs-numerical types and explicitly declares IDs and relationships rather than asking metadata inference to guess them.

In [6]:
TABLE_NAMES = ["district", "client", "account", "disp", "card", "loan", "order", "trans"]
PRIMARY_KEYS = {
    "district": "district_id",
    "client": "client_id",
    "account": "account_id",
    "disp": "disp_id",
    "card": "card_id",
    "loan": "loan_id",
    "order": "order_id",
    "trans": "trans_id",
}
RELATIONSHIPS = [
    ("district", "district_id", "client", "district_id"),
    ("district", "district_id", "account", "district_id"),
    ("client", "client_id", "disp", "client_id"),
    ("account", "account_id", "disp", "account_id"),
    ("disp", "disp_id", "card", "disp_id"),
    ("account", "account_id", "loan", "account_id"),
    ("account", "account_id", "order", "account_id"),
    ("account", "account_id", "trans", "account_id"),
]

tables = {name: pd.read_csv(BERKA_SOURCE / f"{name}.csv") for name in TABLE_NAMES}
domains = {
    name: json.loads((BERKA_SOURCE / f"{name}_domain.json").read_text())
    for name in TABLE_NAMES
}

for table_name, primary_key in PRIMARY_KEYS.items():
    if tables[table_name][primary_key].isna().any() or tables[table_name][primary_key].duplicated().any():
        raise ValueError(f"{table_name}.{primary_key} must be present and unique.")

for parent_table, parent_key, child_table, foreign_key in RELATIONSHIPS:
    if tables[child_table][foreign_key].isna().any():
        raise ValueError(f"{child_table}.{foreign_key} contains missing foreign keys.")
    invalid_keys = set(tables[child_table][foreign_key]) - set(tables[parent_table][parent_key])
    if invalid_keys:
        raise ValueError(f"{child_table}.{foreign_key} contains keys absent from {parent_table}.{parent_key}.")
{name: frame.shape for name, frame in tables.items()}#, index=["rows, columns"]

{'district': (77, 16),
 'client': (5369, 6),
 'account': (4500, 4),
 'disp': (5369, 4),
 'card': (892, 4),
 'loan': (682, 7),
 'order': (6471, 6),
 'trans': (100000, 10)}

In [7]:
metadata = Metadata()
metadata.detect_from_dataframes(tables)

# Replace any automatically inferred relationships with the explicit Berka schema above.
for relationship in metadata.relationships.copy():
    metadata.remove_relationship(
        parent_table_name=relationship["parent_table_name"],
        child_table_name=relationship["child_table_name"],
    )

key_columns = {
    table: {PRIMARY_KEYS[table]} for table in TABLE_NAMES
}
for _, _, child_table, foreign_key in RELATIONSHIPS:
    key_columns[child_table].add(foreign_key)

for table_name, frame in tables.items():
    metadata.set_primary_key(table_name, PRIMARY_KEYS[table_name])
    for column in frame.columns:
        if column in key_columns[table_name]:
            metadata.update_column(table_name, column, sdtype="id")
            continue

        domain_type = domains[table_name].get(column, {}).get("type")
        if domain_type == "discrete":
            metadata.update_column(table_name, column, sdtype="categorical")
        elif domain_type == "continuous":
            metadata.update_column(table_name, column, sdtype="numerical")
            representation = "Int64" if pd.api.types.is_integer_dtype(frame[column]) else "Float"
            metadata.update_column(
                table_name,
                column,
                computer_representation=representation,
            )
        else:
            raise ValueError(f"No Berka domain type declared for {table_name}.{column}.")

for parent_table, parent_key, child_table, foreign_key in RELATIONSHIPS:
    metadata.add_relationship(
        parent_table_name=parent_table,
        child_table_name=child_table,
        parent_primary_key=parent_key,
        child_foreign_key=foreign_key,
    )

metadata.validate()
metadata.validate_data(tables)
print("Tables and explicit Berka relationships passed Syntherela metadata validation.")

/home/coder/RelDiff/.venv/lib/python3.10/site-packages/sdv/metadata/single_table.py:937: UserWarning: There is an existing primary key 'district_id'. This key will be removed.
  warnings.warn(
/home/coder/RelDiff/.venv/lib/python3.10/site-packages/sdv/metadata/single_table.py:937: UserWarning: There is an existing primary key 'client_id'. This key will be removed.
  warnings.warn(
/home/coder/RelDiff/.venv/lib/python3.10/site-packages/sdv/metadata/single_table.py:937: UserWarning: There is an existing primary key 'account_id'. This key will be removed.
  warnings.warn(
/home/coder/RelDiff/.venv/lib/python3.10/site-packages/sdv/metadata/single_table.py:937: UserWarning: There is an existing primary key 'disp_id'. This key will be removed.
  warnings.warn(
/home/coder/RelDiff/.venv/lib/python3.10/site-packages/sdv/metadata/single_table.py:937: UserWarning: There is an existing primary key 'card_id'. This key will be removed.
  warnings.warn(
/home/coder/RelDiff/.venv/lib/python3.10/site-

Tables and explicit Berka relationships passed Syntherela metadata validation.


## Save a RelDiff input dataset

This uses a separate dataset name under `RelDiff/data/original/` and refuses to overwrite an existing dataset. If rerunning the setup cell, choose a new `DATASET_NAME` or reuse the already-prepared files without rerunning the save cell.

In [8]:
if ORIGINAL_DIR.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing RelDiff input data at {ORIGINAL_DIR}. "
        "Use the existing metadata/data or choose a new DATASET_NAME."
    )

ORIGINAL_DIR.mkdir(parents=True, exist_ok=False)
save_tables(
    tables,
    str(ORIGINAL_DIR),
    metadata=metadata,
    save_metadata=True,
)
print("Saved input tables and metadata to:", ORIGINAL_DIR)

TypeError: save_tables() got an unexpected keyword argument 'metadata'

## Train RelDiff

The checked-in trainer's checkpoint threshold is epoch 4,001. Set a unique `RUN_ID` for each experiment because checkpoints are stored under that run-specific directory. Reduce batch size if CUDA memory is insufficient; increase training epochs for a more serious quality comparison.

In [ ]:
NUM_EPOCHS = 5_000
BATCH_SIZE = 1_024
SAMPLING_BATCH_SIZE = 10_000
NUM_WORKERS = min(8, os.cpu_count() or 1)
if NUM_EPOCHS <= 4_000:
    raise ValueError("RelDiff only saves the best sampling checkpoint after epoch 4,000.")

train_command = [
    sys.executable,
    str(RELDIFF_ROOT / "src/scripts/train_joint_diffusion.py"),
    DATASET_NAME,
    "--num-epochs", str(NUM_EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--sampling-batch-size", str(SAMPLING_BATCH_SIZE),
    "--run-id", RUN_ID,
    "--config-path", str(MODEL_CONFIG),
    "--dataset-config-path", str(DATASET_CONFIG),
    "--device", "cuda",
    "--sampling-device", "cuda",
    "--num-workers", str(NUM_WORKERS),
    "--no-wandb",
]
print(" ".join(train_command))
subprocess.run(train_command, cwd=RELDIFF_ROOT, check=True)

## Sample a synthetic relational database

Sampling loads the best non-EMA checkpoint produced during training and uses the original Berka graph structure. The default result directory is `data/synthetic/<dataset>/RelDiff/<run_id>/sample1/`.

In [ ]:
checkpoint_dir = RELDIFF_ROOT / "ckpt" / DATASET_NAME / f"multi{RUN_ID}"
best_checkpoints = list(checkpoint_dir.glob("best_model_*.pt"))
if not best_checkpoints:
    raise FileNotFoundError(
        f"No best-model checkpoint found in {checkpoint_dir}. Confirm training completed beyond epoch 4,000."
    )

sample_command = [
    sys.executable,
    str(RELDIFF_ROOT / "src/scripts/sample_joint_diffusion.py"),
    DATASET_NAME,
    "--run-id", RUN_ID,
    "--num-samples", "1",
    "--sampling-batch-size", str(SAMPLING_BATCH_SIZE),
    "--config-path", str(MODEL_CONFIG),
    "--dataset-config-path", str(DATASET_CONFIG),
    "--device", "cuda",
    "--sampling-device", "cuda",
]
print(" ".join(sample_command))
subprocess.run(sample_command, cwd=RELDIFF_ROOT, check=True)

SYNTHETIC_DIR = (
    RELDIFF_ROOT / "data/synthetic" / DATASET_NAME / "RelDiff" / RUN_ID / "sample1"
)
if not (SYNTHETIC_DIR / "trans.csv").is_file():
    raise FileNotFoundError(f"RelDiff output table not found: {SYNTHETIC_DIR / 'trans.csv'}")
print("Synthetic database:", SYNTHETIC_DIR)

## Inspect synthetic transaction output

Check table cardinalities, required columns, and generated overdraft prevalence before using the data for transfer evaluation. With `--structure original`, table counts are expected to match the source graph.

In [ ]:
synthetic_trans = pd.read_csv(SYNTHETIC_DIR / "trans.csv")
real_trans = tables["trans"]
required_trans_columns = {
    "account_id", "trans_date", "trans_type", "operation", "amount",
    "balance", "k_symbol", "bank",
}
missing_columns = required_trans_columns.difference(synthetic_trans.columns)
if missing_columns:
    raise ValueError(f"Generated trans table is missing columns: {sorted(missing_columns)}")

comparison = pd.DataFrame(
    {
        "real_rows": [len(real_trans)],
        "synthetic_rows": [len(synthetic_trans)],
        "real_overdraft_prevalence": [real_trans["balance"].lt(0).mean()],
        "synthetic_overdraft_prevalence": [synthetic_trans["balance"].lt(0).mean()],
        "synthetic_missing_balance": [synthetic_trans["balance"].isna().sum()],
    },
    index=["RelDiff transaction output"],
)
display(comparison)
display(synthetic_trans.head())

## Use with overdraft transfer evaluation

Point `OverdraftTransferEvaluator.evaluate_synthetic` at `SYNTHETIC_DIR / "trans.csv"`. Use the same real-data representation and category codes the evaluator expects. This output is the ordinary Berka feature set, so do not pass engineered history columns as `additional_features`. If you want to test the feature-engineered experiment, first extend the RelDiff input metadata and tables consistently, retrain, and verify those generated columns; RelDiff will not recompute the engineered features from synthetic transaction sequences.

Compare prevalence, average precision, and ROC-AUC together. The original-structure sample keeps the source transaction count and account relations; a sampling batch-size change only affects inference batching, not the generated database size.